# Study 2: Reward design and pollution pressure

- Notebook 07 showed that most agents stop developing around EDI 70 and then spend much of the episode mitigating. I want to see whether paying for income each step changes that pattern.
- I compare three economic rewards: the current EDI gain, income above the starting income, and both together. Local, global, and mixed sharing still work as before.
- First I hold the environment fixed. Later sections show how regions affect neighbors and prepare a separate weaker-mitigation test. I will read the reward results before running that test.
- This notebook keeps policies and tables in memory. It does not write CSV, JSON, or image files.

## Settings

In [39]:
import random

import pandas as pd

from study_2_economic_sustainability.agents import TabularQLearner
from study_2_economic_sustainability.config import EnvironmentConfig, TrainingConfig
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment

In [40]:
REWARD_DESIGNS = ("edi_gain", "income", "combined")
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))
TRAINING_EPISODES = 2_000
INCOME_REWARD_SCALE = 10.0
environment_config = EnvironmentConfig()
training_config = TrainingConfig()

## Define the economic rewards

- `edi_gain` gives the original reward for EDI gained on the current step.
- `income` gives `(income earned this step - 20) / 10`. It pays 0 at EDI 0 and 5 at EDI 100. I subtract the 20 base income so every action does not receive an unrelated survival bonus.
- `combined` adds both economic terms. This is exploratory: the scale of 10 puts the income bonus near the current middle development gain of 5 at EDI 100. It is not a calibrated economic value.

In [41]:
def economic_reward(info, region, design, config):
    edi_gain = info["development_gains"][region]
    income_bonus = (info["income"][region] - config.income_base) / INCOME_REWARD_SCALE
    if design == "income":
        return income_bonus
    if design == "combined":
        return edi_gain + income_bonus
    return edi_gain

In [42]:
def learning_rewards(environment, original_rewards, info, terminated, design):
    if design == "edi_gain":
        return original_rewards

    config = environment.config
    local_rewards = []
    for region in range(4):
        reward = config.economy_weight * economic_reward(info, region, design, config)
        reward += config.pollution_weight * info["pollution_rewards"][region]
        if not info["valid_actions"][region]:
            reward -= config.invalid_action_penalty
        if terminated:
            reward -= config.failure_penalty
        local_rewards.append(reward)

    shared_reward = sum(local_rewards) / 4
    if environment.reward_mode == "global":
        return (shared_reward,) * 4
    if environment.reward_mode == "mixed":
        return tuple(0.7 * reward + 0.3 * shared_reward for reward in local_rewards)
    return tuple(local_rewards)

## Train matched policies

In [43]:
def choose_actions(agents, observations, greedy):
    actions = []
    for region in range(4):
        actions.append(agents[region].select_action(observations[region], greedy=greedy))
    return tuple(actions)

In [44]:
def train_episode(environment, agents, seed, design):
    observations = environment.reset(seed)
    while True:
        actions = choose_actions(agents, observations, greedy=False)
        next_observations, original_rewards, terminated, truncated, info = environment.step(actions)
        rewards = learning_rewards(environment, original_rewards, info, terminated, design)
        for region in range(4):
            agents[region].update(
                observations[region], actions[region], rewards[region],
                next_observations[region], terminated or truncated,
            )
        observations = next_observations
        if terminated or truncated:
            break

In [45]:
def train_policy(design, reward_mode, training_seed, config):
    environment = EconomicSustainabilityEnvironment(reward_mode, config)
    episode_rng = random.Random(training_seed)
    agents = []
    for region in range(4):
        agents.append(TabularQLearner(training_seed * 10 + region, training_config))
    for episode in range(TRAINING_EPISODES):
        seed = episode_rng.randrange(100_000)
        train_episode(environment, agents, seed, design)
        for agent in agents:
            agent.decay_epsilon()
    return agents

In [46]:
trained_policies = {}
for design in REWARD_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            trained_policies[(design, reward_mode, training_seed)] = train_policy(
                design, reward_mode, training_seed, environment_config
            )
        print(f"Trained {design} with {reward_mode} rewards across five seeds")

Trained edi_gain with local rewards across five seeds
Trained edi_gain with global rewards across five seeds
Trained edi_gain with mixed rewards across five seeds
Trained income with local rewards across five seeds
Trained income with global rewards across five seeds
Trained income with mixed rewards across five seeds
Trained combined with local rewards across five seeds
Trained combined with global rewards across five seeds
Trained combined with mixed rewards across five seeds


## Evaluate the same starts

- GPI is the average global pollution index over an episode.
- `peak_regional_pollution` is the highest load any region reaches. Spending counts valid mitigation purchases only.

In [47]:
def empty_totals():
    return {
        "gpi": 0.0, "spillover": 0.0, "peak_pollution": 0.0,
        "mitigation_spending": 0, "development_actions": 0, "high_actions": 0,
        "last_development_steps": [0, 0, 0, 0],
    }

In [48]:
def add_step_totals(totals, actions, info, step):
    totals["gpi"] += info["global_pollution_index"]
    totals["spillover"] += sum(info["incoming_spillover"])
    totals["peak_pollution"] = max(totals["peak_pollution"], *info["pollution_loads"])
    for region in range(4):
        if actions[region] in (2, 3, 4):
            totals["mitigation_spending"] += info["costs_paid"][region]
        if info["development_gains"][region] > 0:
            totals["development_actions"] += 1
            totals["last_development_steps"][region] = step
        totals["high_actions"] += actions[region] == 4

In [49]:
def episode_result(environment, totals, terminated):
    region_steps = 4 * environment.step_count
    return {
        "steps": environment.step_count,
        "failure_percent": 100 * terminated,
        "mean_gpi": totals["gpi"] / environment.step_count,
        "peak_regional_pollution": totals["peak_pollution"],
        "mitigation_spending": totals["mitigation_spending"],
        "final_edi": sum(environment.development_indices) / 4,
        "final_currency": sum(environment.currency_balances) / 4,
        "development_percent": 100 * totals["development_actions"] / region_steps,
        "last_development_step": sum(totals["last_development_steps"]) / 4,
        "high_percent": 100 * totals["high_actions"] / region_steps,
        "mean_incoming_spillover": totals["spillover"] / region_steps,
    }

In [50]:
def evaluate_episode(agents, reward_mode, seed, config):
    environment = EconomicSustainabilityEnvironment(reward_mode, config)
    observations = environment.reset(seed)
    totals = empty_totals()
    while True:
        actions = choose_actions(agents, observations, greedy=True)
        observations, rewards, terminated, truncated, info = environment.step(actions)
        add_step_totals(totals, actions, info, environment.step_count)
        if terminated or truncated:
            break
    return episode_result(environment, totals, terminated)

In [51]:
episode_rows = []
for design in REWARD_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            agents = trained_policies[(design, reward_mode, training_seed)]
            for environment_seed in EVALUATION_SEEDS:
                row = evaluate_episode(agents, reward_mode, environment_seed, environment_config)
                row.update(reward_design=design, reward_mode=reward_mode, training_seed=training_seed)
                episode_rows.append(row)
episodes = pd.DataFrame(episode_rows)
print(f"Evaluated {len(episodes)} episodes")

Evaluated 900 episodes


## Compare outcomes

- `last_development_step` averages each region's final successful development step. A region that never develops contributes 0. This helps show whether investment continues later than in notebook 07.
- More development is not automatically better. I want to see whether extra EDI is earned without making pollution, spending, or failure unacceptable for this synthetic study.

In [52]:
MEASURES = [
    "mean_gpi", "peak_regional_pollution", "mitigation_spending",
    "final_edi", "final_currency", "failure_percent",
    "development_percent", "last_development_step", "high_percent",
    "mean_incoming_spillover",
]
seed_results = episodes.groupby(
    ["reward_design", "reward_mode", "training_seed"]
)[MEASURES].mean()
mean_results = seed_results.groupby(level=["reward_design", "reward_mode"]).mean()
seed_spread = seed_results.groupby(level=["reward_design", "reward_mode"]).std()

In [53]:
display(mean_results[["final_edi", "development_percent", "last_development_step", "final_currency", "mitigation_spending"]].round(1))

final_edi  development_percent  \
reward_design reward_mode                                   
combined      global            73.5                  6.1   
              local             73.0                  5.8   
              mixed             73.4                  6.2   
edi_gain      global            51.1                  3.9   
              local             70.0                  5.0   
              mixed             68.2                  4.9   
income        global            50.1                  3.4   
              local             70.0                  5.2   
              mixed             63.8                  4.9   

                           last_development_step  final_currency  \
reward_design reward_mode                                          
combined      global                        39.4          5009.3   
              local                         31.8          3714.8   
              mixed                         42.0          4046.3   
edi_gain      global                        22.0          2664.7   
              local                         24.3          1228.6   
              mixed                         24.2          1335.7   
income        global                        18.3          3366.8   
              local                         38.5          5433.8   
              mixed                         43.6          4866.2   

                           mitigation_spending  
reward_design reward_mode                       
combined      global                   20508.8  
              local                    26127.0  
              mixed                    23940.7  
edi_gain      global                   23515.8  
              local                    35933.8  
              mixed                    34726.9  
income        global                   21103.3  
              local                    18105.4  
              mixed                    17980.0

- The combined reward raises final EDI to about 73 in all three modes. The EDI-gain reference ends at 51 for global, 70 for local, and 68 for mixed. Income alone ends at 50, 70, and 64, so paying only for ongoing income does not consistently extend development.
- With the combined reward, the last development step moves later, to about 32-42 instead of 22-24 in the reference. This measure includes 0 for a region that never develops.
- Both new rewards leave more final currency and generally reduce mitigation spending. More currency does not necessarily mean agents found a better balance; the pollution table below matters too.
- The EDI-gain reference was retrained in this notebook with the same terminal-step update rule as the alternatives. Its global result should be compared with these rows, rather than treated as the saved policy from notebook 07.

In [54]:
display(mean_results[["mean_gpi", "peak_regional_pollution", "failure_percent", "high_percent"]].round(1))

mean_gpi  peak_regional_pollution  failure_percent  \
reward_design reward_mode                                                       
combined      global           16.4                     47.3              0.0   
              local            12.5                     46.1              0.0   
              mixed            15.6                     47.6              0.0   
edi_gain      global            7.2                     40.5              0.0   
              local             2.4                     41.2              0.0   
              mixed             2.7                     41.1              0.0   
income        global           12.0                     45.2              0.0   
              local            18.4                     44.0              0.0   
              mixed            16.4                     49.4              0.0   

                           high_percent  
reward_design reward_mode                
combined      global               32.3  
              local                52.8  
              mixed                47.0  
edi_gain      global               39.5  
              local                85.8  
              mixed                81.4  
income        global               32.5  
              local                24.3  
              mixed                21.7

- The combined reward increases mean GPI compared with the EDI-gain reference: global 7.2 to 16.4, local 2.4 to 12.5, and mixed 2.7 to 15.6. Income alone also raises GPI.
- High mitigation falls from 85.8% to 52.8% for local and from 81.4% to 47.0% for mixed under the combined reward. Income alone reduces high even more, but does not reliably raise final EDI.
- None of these 900 held-out episodes fails, and mean peak regional pollution stays below 50. That suggests the current environment still leaves room to test a harder mitigation setting, but it does not establish that higher pollution is desirable.

In [55]:
display(seed_spread[["final_edi", "mean_gpi", "mitigation_spending", "failure_percent"]].round(1))

final_edi  mean_gpi  mitigation_spending  \
reward_design reward_mode                                             
combined      global             6.3       8.9               7321.4   
              local              4.1       5.1               2360.1   
              mixed             10.6       4.9               3450.9   
edi_gain      global            21.4       5.2               4886.0   
              local              0.0       0.0               1249.6   
              mixed              4.0       0.4               2823.3   
income        global             4.8       6.2               2379.7   
              local              5.3       3.5               2975.3   
              mixed             14.4       9.3               3801.7   

                           failure_percent  
reward_design reward_mode                   
combined      global                   0.0  
              local                    0.0  
              mixed                    0.0  
edi_gain      global                   0.0  
              local                    0.0  
              mixed                    0.0  
income        global                   0.0  
              local                    0.0  
              mixed                    0.0

- These standard deviations are across five training seeds, after averaging each seed's 20 evaluation episodes. Global EDI-gain has a large final-EDI spread of 21.4, while combined global is 6.3. The combined result is more consistent on that measure.
- Pollution and spending still vary by seed, especially combined global GPI (8.9) and spending (7,321). I would not choose a reward solely from the mean table.

## How do regions affect each other?

- In the current environment, a region's **remaining emissions after mitigation** spill into its two neighbors. Each neighbor receives 20% of those remaining emissions on that step. The region's stored pollution load does not spread directly.
- For example, five extra units of remaining emissions in region 0 add five units to its own new load and one unit to each of regions 1 and 2. Region 3 gets none directly. This raises GPI by 1.75 units before any later policy response.
- A neighbor may change its later actions because it observes its own pollution and GPI. That indirect effect needs episode analysis; the next table shows only the immediate physical effect.

In [56]:
EXTRA_REMAINING_EMISSIONS = 5
spillover_rows = []
for source_region in range(4):
    row = {"source_region": source_region}
    for affected_region in range(4):
        if source_region == affected_region:
            change = EXTRA_REMAINING_EMISSIONS
        elif source_region in environment_config.neighbors[affected_region]:
            change = environment_config.spillover * EXTRA_REMAINING_EMISSIONS
        else:
            change = 0
        row[f"region_{affected_region}_load_change"] = change
    spillover_rows.append(row)
display(pd.DataFrame(spillover_rows).set_index("source_region"))

,region_0_load_change,region_1_load_change,region_2_load_change,region_3_load_change
source_region,,,,
0,5.0,1.0,1.0,0.0
1,1.0,5.0,0.0,1.0
2,1.0,0.0,5.0,1.0
3,0.0,1.0,1.0,5.0


### Spillover under the learned policies

- `mean_incoming_spillover` is the average amount a region receives from its two neighbors per step. Zero means the neighbors' remaining emissions were fully mitigated. This is an outcome of the joint policies, not a fixed property of the reward formula.

In [57]:
display(mean_results[["mean_incoming_spillover"]].round(2))

mean_incoming_spillover
reward_design reward_mode                         
combined      global                          1.56
              local                           1.16
              mixed                           1.46
edi_gain      global                          0.70
              local                           0.26
              mixed                           0.29
income        global                          1.22
              local                           1.79
              mixed                           1.65

- Incoming spillover rises under the new rewards. For local policies it is 0.26 per region-step with EDI gain, 1.16 with the combined reward, and 1.79 with income alone. Mixed policies show the same direction.
- The new policies leave more emissions after mitigation, so neighbors receive more pollution even when the source region's own load stays manageable. This is one reason to compare each region's outcomes, not just GPI.
- This table still describes the current mitigation capacity. The separate weaker-high results appear at the end of the notebook.

## What if mitigation is weaker?

- As a separate stress test, I would like to reduce only high mitigation capacity from 30 to 20. Medium stays at 15, and prices stay fixed. This makes the extra 25 currency for high buy five more units of capacity instead of 15.
- I am choosing weaker mitigation rather than higher base emissions because changing base emissions would also raise pollution at EDI 0.

In [58]:
stressed_config = EnvironmentConfig()
stressed_config.mitigation_capacity = (0, 0, 5, 15, 20)
capacity_rows = []
for name, settings in (("current", environment_config), ("weaker_high", stressed_config)):
    capacity_rows.append({
        "setting": name,
        "medium_capacity": settings.mitigation_capacity[3],
        "high_capacity": settings.mitigation_capacity[4],
        "high_extra_over_medium": settings.mitigation_capacity[4] - settings.mitigation_capacity[3],
    })
display(pd.DataFrame(capacity_rows).set_index("setting"))

,medium_capacity,high_capacity,high_extra_over_medium
setting,,,
current,15,30,15
weaker_high,15,20,5


### Run the capacity test after reviewing the reward results

- The first Run All used `RUN_STRESS = False` to answer the reward question alone. The capacity test was then run with `RUN_STRESS = True`, adding another 45 training runs.
- The comparison below subtracts the current-capacity result from the weaker-high result within each reward design, reward mode, and training seed.

In [59]:
RUN_STRESS = True

In [60]:
stressed_policies = {}
if RUN_STRESS:
    for design in REWARD_DESIGNS:
        for reward_mode in REWARD_MODES:
            for training_seed in TRAINING_SEEDS:
                stressed_policies[(design, reward_mode, training_seed)] = train_policy(
                    design, reward_mode, training_seed, stressed_config
                )
            print(f"Trained weaker-high {design}, {reward_mode} across five seeds")

Trained weaker-high edi_gain, local across five seeds
Trained weaker-high edi_gain, global across five seeds
Trained weaker-high edi_gain, mixed across five seeds
Trained weaker-high income, local across five seeds
Trained weaker-high income, global across five seeds
Trained weaker-high income, mixed across five seeds
Trained weaker-high combined, local across five seeds
Trained weaker-high combined, global across five seeds
Trained weaker-high combined, mixed across five seeds


In [61]:
stressed_rows = []
if RUN_STRESS:
    for design in REWARD_DESIGNS:
        for reward_mode in REWARD_MODES:
            for training_seed in TRAINING_SEEDS:
                agents = stressed_policies[(design, reward_mode, training_seed)]
                for environment_seed in EVALUATION_SEEDS:
                    row = evaluate_episode(agents, reward_mode, environment_seed, stressed_config)
                    row.update(reward_design=design, reward_mode=reward_mode, training_seed=training_seed)
                    stressed_rows.append(row)
    stressed_episodes = pd.DataFrame(stressed_rows)
    print(f"Evaluated {len(stressed_episodes)} weaker-high episodes")

Evaluated 900 weaker-high episodes


In [62]:
if RUN_STRESS:
    stressed_seed_results = stressed_episodes.groupby(
        ["reward_design", "reward_mode", "training_seed"]
    )[MEASURES].mean()
    capacity_change = stressed_seed_results - seed_results
    capacity_change_mean = capacity_change.groupby(
        level=["reward_design", "reward_mode"]
    ).mean()
    display(capacity_change_mean[[
        "mean_gpi", "peak_regional_pollution", "failure_percent",
        "final_edi", "last_development_step", "mitigation_spending",
        "high_percent",
        "mean_incoming_spillover",
    ]].round(1))

mean_gpi  peak_regional_pollution  failure_percent  \
reward_design reward_mode                                                       
combined      global           -2.4                     -1.4              0.0   
              local             3.9                      0.5              0.0   
              mixed             2.6                     -0.8              0.0   
edi_gain      global            0.2                      0.1              0.0   
              local             0.5                      0.0              0.0   
              mixed             0.9                     -0.1              0.0   
income        global           -0.9                      0.6              0.0   
              local            -1.3                     -1.2              0.0   
              mixed            -3.9                     -6.1              0.0   

                           final_edi  last_development_step  \
reward_design reward_mode                                     
combined      global            -1.2                    1.7   
              local              3.0                    4.5   
              mixed             -1.8                  -14.3   
edi_gain      global            -0.2                    2.0   
              local             -6.0                   -1.7   
              mixed             -4.9                   -1.1   
income        global            -3.2                   -2.0   
              local             -3.5                   -0.0   
              mixed              0.1                  -10.8   

                           mitigation_spending  high_percent  \
reward_design reward_mode                                      
combined      global                    1936.2           1.9   
              local                     -927.4          -3.5   
              mixed                    -1167.6          -5.8   
edi_gain      global                   -1407.9          -5.6   
              local                    -1175.2          -3.8   
              mixed                    -3005.6         -12.9   
income        global                     739.0          -0.9   
              local                     1273.0          -0.6   
              mixed                     2575.2           1.4   

                           mean_incoming_spillover  
reward_design reward_mode                           
combined      global                          -0.4  
              local                            0.1  
              mixed                            0.0  
edi_gain      global                           0.1  
              local                            0.0  
              mixed                            0.0  
income        global                          -0.2  
              local                           -0.4  
              mixed                           -0.6

- Each value is weaker-high capacity minus current capacity for the same reward design, mode, and training seed, then averaged across seeds. A positive GPI difference means more pollution.
- The response is mixed. Under the combined reward, local GPI rises by 3.9 and final EDI rises by 3.0, while global GPI falls by 2.4 and spending rises by about 1,936. Mixed GPI rises by 2.6 while final EDI falls by 1.8.
- Under the original EDI-gain reward, local and mixed final EDI fall by 6.0 and 4.9. Income-only mixed GPI falls by 3.9 even though high mitigation is weaker, while mitigation spending rises by about 2,575. Retraining lets agents change their actions, so lower capacity does not guarantee higher GPI.
- None of the evaluated stress episodes fails. Reducing high capacity from 30 to 20 changes behavior but still does not create failure pressure in these held-out starts. I would inspect seed-level differences and regional outcomes before calling any setting better.

## Does the capacity result hold across seeds?

- The earlier difference table averages five matched training seeds. I want to see whether one seed drives the apparent improvement or deterioration.
- The first table shows each combined-reward seed. The second shows the standard deviation of paired differences for all reward designs. Positive differences mean weaker high mitigation increased the measure.

In [63]:
display(capacity_change.loc["combined", ["mean_gpi", "final_edi", "mitigation_spending", "high_percent"]].round(1))

mean_gpi  final_edi  mitigation_spending  \
reward_mode training_seed                                             
global      11                 15.2       -0.1              -8984.5   
            23                 -3.0       12.4               8113.0   
            37                 -6.9       -9.9                -61.8   
            41                 -3.8       -8.3               4463.2   
            53                -13.6       -0.3               6150.8   
local       11                  2.2        7.5               2867.8   
            23                  0.2        0.0                493.8   
            37                 11.2       15.0              -6480.0   
            41                 12.8        0.0              -4933.0   
            53                 -6.8       -7.5               3414.2   
mixed       11                  8.4       13.1              -7368.0   
            23                 -1.6        0.0               3066.0   
            37                 -2.4       -7.5               1347.0   
            41                  4.3       -7.5               1278.8   
            53                  4.2       -7.3              -4161.5   

                           high_percent  
reward_mode training_seed                
global      11                    -25.4  
            23                     35.6  
            37                    -14.7  
            41                     13.4  
            53                      0.7  
local       11                     15.8  
            23                      0.5  
            37                    -25.8  
            41                    -10.2  
            53                      2.1  
mixed       11                    -31.4  
            23                      7.5  
            37                     -3.0  
            41                     13.3  
            53                    -15.6

- The combined-reward capacity effect changes direction across seeds. For global reward, four seeds lower GPI while seed 11 raises it by 15.2. Local has four higher-GPI seeds and one lower-GPI seed; mixed also has both directions.
- The mean capacity difference therefore hides substantial variation. I would not use the weaker-high setting as evidence that pollution must rise, or that a particular reward mode consistently adapts better.

In [64]:
capacity_seed_spread = capacity_change.groupby(
    level=["reward_design", "reward_mode"]
)[["mean_gpi", "final_edi", "mitigation_spending"]].std()
display(capacity_seed_spread.round(1))

mean_gpi  final_edi  mitigation_spending
reward_design reward_mode                                          
combined      global           10.7        8.8               6810.9
              local             8.1        8.6               4531.8
              mixed             4.5        9.0               4405.7
edi_gain      global            7.6       35.1               9861.5
              local             0.4        7.7               1553.2
              mixed             1.8       11.3               4088.5
income        global           11.0        7.9               4941.4
              local             6.0        6.1               3169.0
              mixed            10.4       14.1               5717.9

- For the combined reward, the seed standard deviation of the GPI change is 10.7 under global, 8.1 under local, and 4.5 under mixed. Each is larger than its mean GPI change in the earlier table.
- This supports treating the weaker-high result as an exploratory stress check, not a stable improvement to the environment.

## Which regions bear the pollution?

- I compare each region under the current and weaker-high capacities. `remaining_emissions` are that region's emissions after its own mitigation; these are what enter its neighbors' pollution updates.
- `incoming_spillover` is pollution received from the two neighbors. The region's own pollution load includes its old pollution, its remaining emissions, and incoming spillover.
- This evaluation reuses policies already in memory. It does not retrain them or write files.

In [65]:
def empty_region_totals():
    return {
        "pollution_load": [0.0] * 4,
        "remaining_emissions": [0.0] * 4,
        "incoming_spillover": [0.0] * 4,
    }

In [66]:
def add_region_step(totals, info):
    for region in range(4):
        totals["pollution_load"][region] += info["pollution_loads"][region]
        totals["remaining_emissions"][region] += info["remaining_emissions"][region]
        totals["incoming_spillover"][region] += info["incoming_spillover"][region]

In [67]:
def evaluate_regions(agents, reward_mode, seed, config):
    environment = EconomicSustainabilityEnvironment(reward_mode, config)
    observations = environment.reset(seed)
    totals = empty_region_totals()
    while True:
        actions = choose_actions(agents, observations, greedy=True)
        observations, rewards, terminated, truncated, info = environment.step(actions)
        add_region_step(totals, info)
        if terminated or truncated:
            break

    rows = []
    for region in range(4):
        rows.append({
            "region": region,
            "mean_pollution_load": totals["pollution_load"][region] / environment.step_count,
            "mean_remaining_emissions": totals["remaining_emissions"][region] / environment.step_count,
            "mean_incoming_spillover": totals["incoming_spillover"][region] / environment.step_count,
            "final_edi": environment.development_indices[region],
        })
    return rows

In [68]:
region_rows = []
if RUN_STRESS:
    for setting, config, policies in (
        ("current", environment_config, trained_policies),
        ("weaker_high", stressed_config, stressed_policies),
    ):
        for design in REWARD_DESIGNS:
            for reward_mode in REWARD_MODES:
                for training_seed in TRAINING_SEEDS:
                    agents = policies[(design, reward_mode, training_seed)]
                    for environment_seed in EVALUATION_SEEDS:
                        rows = evaluate_regions(agents, reward_mode, environment_seed, config)
                        for row in rows:
                            row.update(
                                setting=setting, design=design,
                                reward_mode=reward_mode, training_seed=training_seed,
                            )
                        region_rows.extend(rows)
    regional_episodes = pd.DataFrame(region_rows)
    print(f"Recorded {len(regional_episodes)} region-episodes")

Recorded 7200 region-episodes


In [69]:
if RUN_STRESS:
    REGION_MEASURES = [
        "mean_pollution_load", "mean_remaining_emissions",
        "mean_incoming_spillover", "final_edi",
    ]
    region_seed = regional_episodes.groupby(
        ["setting", "design", "reward_mode", "training_seed", "region"]
    )[REGION_MEASURES].mean()
    region_change = region_seed.loc["weaker_high"] - region_seed.loc["current"]
    region_mean_change = region_change.groupby(
        level=["design", "reward_mode", "region"]
    ).mean()
    display(region_mean_change.loc["combined"].round(2))

mean_pollution_load  mean_remaining_emissions  \
reward_mode region                                                  
global      0                     -4.05                     -2.14   
            1                     -3.72                     -1.59   
            2                     -4.86                     -0.80   
            3                      3.03                      0.51   
local       0                      7.75                      0.49   
            1                      9.54                      1.89   
            2                      2.75                      0.04   
            3                     -4.45                     -0.99   
mixed       0                     13.21                      2.62   
            1                     -1.12                     -0.95   
            2                     -3.22                     -1.23   
            3                      1.36                     -0.44   

                    mean_incoming_spillover  final_edi  
reward_mode region                                      
global      0                         -0.48      -6.00  
            1                         -0.32      12.00  
            2                         -0.32       0.96  
            3                         -0.48     -11.88  
local       0                          0.39       6.00  
            1                         -0.10       6.00  
            2                         -0.10      -6.00  
            3                          0.39       6.00  
mixed       0                         -0.44       0.00  
            1                          0.44       5.50  
            2                          0.44       5.16  
            3                         -0.44     -18.00

- This table shows the weaker-high minus current difference for each region under the combined reward, averaged across five matched training seeds. Positive incoming spillover means that region receives more pollution from its neighbors.
- The burden moves unevenly. Under the combined mixed reward, region 0's mean load rises by 13.21 and its remaining emissions rise by 2.62, while its neighbors 1 and 2 each receive 0.44 more spillover. Their total loads still fall by 1.12 and 3.22, showing that each region's own emissions and mitigation also matter.
- Under combined local reward, regions 0 and 1 have higher loads, while region 3 has a lower load. Under global reward, three regions have lower loads and region 3 has a higher load. GPI alone would hide these shifts.

## What if development produces more emissions?

- The earlier capacity test weakened one mitigation action. Here I restore the original capacities and change only how emissions rise with EDI.
- The current formula is `8 * (1 + EDI / 100)`. This test uses `8 * (1 + 2 * EDI / 100)`: both start at 8 emissions when EDI is 0, but the test reaches 24 instead of 16 at EDI 100.
- I retrain the current EDI-gain and combined rewards for all three sharing modes, using the same five training seeds and 20 evaluation starts. This adds 30 training runs. I leave income-only aside because it did not consistently extend development in the first comparison.
- Restart the notebook kernel and use Run All before reading these results. An existing kernel may still hold the older environment code without the new emissions setting.

In [70]:
emissions_config = EnvironmentConfig()
emissions_config.emissions_edi_multiplier = 2.0
SLOPE_DESIGNS = ("edi_gain", "combined")
emissions_preview = []
for edi in (0, 70, 100):
    emissions_preview.append({
        "edi": edi,
        "current_emissions": environment_config.emissions_base * (
            1 + environment_config.emissions_edi_multiplier * edi / 100
        ),
        "higher_slope_emissions": emissions_config.emissions_base * (
            1 + emissions_config.emissions_edi_multiplier * edi / 100
        ),
    })
display(pd.DataFrame(emissions_preview).set_index("edi"))

,current_emissions,higher_slope_emissions
edi,,
0,8.0,8.0
70,13.6,19.2
100,16.0,24.0


### Train with the higher EDI emissions slope

In [71]:
emissions_policies = {}
for design in SLOPE_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            emissions_policies[(design, reward_mode, training_seed)] = train_policy(
                design, reward_mode, training_seed, emissions_config
            )
        print(f"Trained higher-slope {design}, {reward_mode} across five seeds")

Trained higher-slope edi_gain, local across five seeds
Trained higher-slope edi_gain, global across five seeds
Trained higher-slope edi_gain, mixed across five seeds
Trained higher-slope combined, local across five seeds
Trained higher-slope combined, global across five seeds
Trained higher-slope combined, mixed across five seeds


### Evaluate matched starts

In [72]:
emissions_rows = []
for design in SLOPE_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            agents = emissions_policies[(design, reward_mode, training_seed)]
            for environment_seed in EVALUATION_SEEDS:
                row = evaluate_episode(agents, reward_mode, environment_seed, emissions_config)
                row.update(reward_design=design, reward_mode=reward_mode, training_seed=training_seed)
                emissions_rows.append(row)
emissions_episodes = pd.DataFrame(emissions_rows)
print(f"Evaluated {len(emissions_episodes)} higher-slope episodes")

Evaluated 600 higher-slope episodes


In [73]:
emissions_seed_results = emissions_episodes.groupby(
    ["reward_design", "reward_mode", "training_seed"]
)[MEASURES].mean()
base_slope_seed = seed_results[
    seed_results.index.get_level_values("reward_design").isin(SLOPE_DESIGNS)
]
emissions_change = emissions_seed_results - base_slope_seed
emissions_mean = emissions_seed_results.groupby(
    level=["reward_design", "reward_mode"]
).mean()
display(emissions_mean[[
    "final_edi", "last_development_step", "mean_gpi",
    "peak_regional_pollution", "failure_percent",
    "mitigation_spending", "mean_incoming_spillover",
]].round(1))

final_edi  last_development_step  mean_gpi  \
reward_design reward_mode                                               
combined      global            71.2                   42.0      17.2   
              local             76.0                   47.6      13.4   
              mixed             74.5                   47.1      13.3   
edi_gain      global            48.9                   18.3       5.0   
              local             67.4                   26.3       3.5   
              mixed             66.7                   25.4       3.5   

                           peak_regional_pollution  failure_percent  \
reward_design reward_mode                                             
combined      global                          52.3              0.0   
              local                           54.0              0.0   
              mixed                           52.0              0.0   
edi_gain      global                          44.4              0.0   
              local                           45.8              0.0   
              mixed                           46.0              0.0   

                           mitigation_spending  mean_incoming_spillover  
reward_design reward_mode                                                
combined      global                   27986.3                      1.6  
              local                    30463.8                      1.3  
              mixed                    30766.0                      1.3  
edi_gain      global                   30433.4                      0.5  
              local                    35785.0                      0.4  
              mixed                    35607.7                      0.4

- With steeper EDI emissions, the combined reward reaches final EDI 71-76 and keeps development going until about step 42-48. The EDI-gain reward ends at EDI 49-67 and around step 18-26. The combined reward still gives agents a stronger reason to develop.
- Combined policies have mean GPI 13-17 and mean peak regional load around 52-54. EDI-gain policies keep GPI around 3-5. No held-out episode fails in either reward design.
- The combined reward therefore buys more development at the cost of more pollution and incoming spillover. This is a tradeoff we can discuss, but the failure threshold of 100 still does not distinguish these policies.

In [74]:
emissions_change_mean = emissions_change.groupby(
    level=["reward_design", "reward_mode"]
).mean()
display(emissions_change_mean[[
    "final_edi", "mean_gpi", "failure_percent",
    "mitigation_spending", "high_percent",
    "mean_incoming_spillover",
]].round(1))

final_edi  mean_gpi  failure_percent  \
reward_design reward_mode                                         
combined      global            -2.3       0.8              0.0   
              local              3.0       1.0              0.0   
              mixed              1.2      -2.3              0.0   
edi_gain      global            -2.1      -2.2              0.0   
              local             -2.6       1.1              0.0   
              mixed             -1.5       0.8              0.0   

                           mitigation_spending  high_percent  \
reward_design reward_mode                                      
combined      global                    7477.4          25.6   
              local                     4336.8          14.9   
              mixed                     6825.3          22.5   
edi_gain      global                    6917.6          26.3   
              local                     -148.8          -0.4   
              mixed                      880.8           3.2   

                           mean_incoming_spillover  
reward_design reward_mode                           
combined      global                           0.1  
              local                            0.2  
              mixed                           -0.1  
edi_gain      global                          -0.2  
              local                            0.1  
              mixed                            0.1

- Each value is the higher-slope outcome minus the current-emissions outcome for the same design, reward mode, and training seed.
- Steeper emissions do not consistently raise GPI after retraining: combined global rises by 0.8, local by 1.0, and mixed falls by 2.3. Combined policies use high mitigation 15-26 percentage points more and spend about 4,300-7,500 more currency.
- I suspect the policies compensate for higher emissions by mitigating more. The higher slope creates a stronger mitigation demand, but it has not produced failures or a clear improvement in the study's tradeoff.

In [75]:
emissions_change_spread = emissions_change.groupby(
    level=["reward_design", "reward_mode"]
)[["final_edi", "mean_gpi", "mitigation_spending"]].std()
display(emissions_change_spread.round(1))

final_edi  mean_gpi  mitigation_spending
reward_design reward_mode                                          
combined      global             3.8      12.1               8232.6
              local              6.7       5.6               2622.0
              mixed              7.4       9.2               5936.2
edi_gain      global            20.2       5.5               6679.2
              local              4.4       0.3               1978.5
              mixed              6.6       0.6               2787.8

- The combined reward's GPI change has seed standard deviation 12.1 for global, 5.6 for local, and 9.2 for mixed. These are larger than the corresponding mean changes of 0.8, 1.0, and -2.3.
- With only five seeds, I would treat the apparent direction of each GPI change as uncertain. The higher slope also does not create failures in the held-out starts.

## Which design should we keep?

- This final table puts the original and higher-slope environments beside one another for the two rewards tested in both. Each row is a mean over five training seeds.
- I would look for development that continues when it is useful, mitigation that responds to pollution, manageable spillover and failures, and results that do not depend heavily on one seed. The study does not need a policy that always reaches EDI 100.
- I will fill in the conclusion after the new outputs are visible.

In [76]:
current_candidates = base_slope_seed.reset_index()
current_candidates["environment"] = "current"
higher_candidates = emissions_seed_results.reset_index()
higher_candidates["environment"] = "higher_edi_emissions"
candidates = pd.concat([current_candidates, higher_candidates], ignore_index=True)
candidate_means = candidates.groupby(
    ["environment", "reward_design", "reward_mode"]
)[MEASURES].mean()
display(candidate_means[[
    "final_edi", "last_development_step", "mean_gpi",
    "failure_percent", "mitigation_spending",
    "mean_incoming_spillover",
]].round(1))

final_edi  \
environment          reward_design reward_mode              
current              combined      global            73.5   
                                   local             73.0   
                                   mixed             73.4   
                     edi_gain      global            51.1   
                                   local             70.0   
                                   mixed             68.2   
higher_edi_emissions combined      global            71.2   
                                   local             76.0   
                                   mixed             74.5   
                     edi_gain      global            48.9   
                                   local             67.4   
                                   mixed             66.7   

                                                last_development_step  \
environment          reward_design reward_mode                          
current              combined      global                        39.4   
                                   local                         31.8   
                                   mixed                         42.0   
                     edi_gain      global                        22.0   
                                   local                         24.3   
                                   mixed                         24.2   
higher_edi_emissions combined      global                        42.0   
                                   local                         47.6   
                                   mixed                         47.1   
                     edi_gain      global                        18.3   
                                   local                         26.3   
                                   mixed                         25.4   

                                                mean_gpi  failure_percent  \
environment          reward_design reward_mode                              
current              combined      global           16.4              0.0   
                                   local            12.5              0.0   
                                   mixed            15.6              0.0   
                     edi_gain      global            7.2              0.0   
                                   local             2.4              0.0   
                                   mixed             2.7              0.0   
higher_edi_emissions combined      global           17.2              0.0   
                                   local            13.4              0.0   
                                   mixed            13.3              0.0   
                     edi_gain      global            5.0              0.0   
                                   local             3.5              0.0   
                                   mixed             3.5              0.0   

                                                mitigation_spending  \
environment          reward_design reward_mode                        
current              combined      global                   20508.8   
                                   local                    26127.0   
                                   mixed                    23940.7   
                     edi_gain      global                   23515.8   
                                   local                    35933.8   
                                   mixed                    34726.9   
higher_edi_emissions combined      global                   27986.3   
                                   local                    30463.8   
                                   mixed                    30766.0   
                     edi_gain      global                   30433.4   
                                   local                    35785.0   
                                   mixed                    35607.7   

                                                mean_incoming_spillover  
environment          r

- My working choice is the **combined EDI-gain plus ongoing-income reward with the original emissions and mitigation settings**. It extends development beyond the EDI-gain reference across all three sharing modes, reduces reliance on high mitigation, and keeps the environmental outcomes visible rather than near zero. I would keep local, global, and mixed as comparison conditions.
- Income alone did not consistently extend development. Weakening high mitigation produced unstable changes across seeds and regions. Doubling the EDI emissions slope made agents spend and mitigate more, without consistently raising GPI or producing failures, so I would not adopt either dynamics change from these results.
- This is a provisional design choice for the synthetic study, not a claim that the reward is optimal. It still rests on five training seeds, and no evaluated policy failed. I would report the original EDI-gain reward as a reference and check the chosen design with more seeds before a final comparison.

## Confirm the reward choice on new seeds

- The five-seed results above helped me choose the combined reward. I now compare it once with the EDI-gain reference using **20 new training seeds** and **20 new evaluation starts**. I will not adjust the income scale after seeing this confirmation.
- Both rewards use the original emissions and mitigation settings, 2,000 training episodes per seed, and identical training and evaluation seeds within each reward mode. Local, global, and mixed remain separate sharing conditions.
- This section trains 120 new policies and evaluates 2,400 episodes. It keeps policies and tables in memory and writes no files.
- I will favor the combined reward if higher EDI holds across most seeds without unacceptable increases in pollution, failures, or uneven regional burdens. The original reward remains a reference even if the combined reward is adopted.

In [77]:
CONFIRMATION_DESIGNS = ("edi_gain", "combined")
CONFIRMATION_TRAINING_SEEDS = tuple(range(101, 121))
CONFIRMATION_EVALUATION_SEEDS = tuple(range(200_000, 200_020))
print(
    f"{len(CONFIRMATION_TRAINING_SEEDS)} new training seeds and "
    f"{len(CONFIRMATION_EVALUATION_SEEDS)} new evaluation starts"
)

20 new training seeds and 20 new evaluation starts


### Train the confirmation policies

In [78]:
confirmation_policies = {}
for design in CONFIRMATION_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in CONFIRMATION_TRAINING_SEEDS:
            confirmation_policies[(design, reward_mode, training_seed)] = train_policy(
                design, reward_mode, training_seed, environment_config
            )
        print(f"Trained confirmation {design}, {reward_mode} across 20 seeds")

Trained confirmation edi_gain, local across 20 seeds
Trained confirmation edi_gain, global across 20 seeds
Trained confirmation edi_gain, mixed across 20 seeds
Trained confirmation combined, local across 20 seeds
Trained confirmation combined, global across 20 seeds
Trained confirmation combined, mixed across 20 seeds


### Evaluate fresh starts

In [79]:
confirmation_rows = []
for design in CONFIRMATION_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in CONFIRMATION_TRAINING_SEEDS:
            agents = confirmation_policies[(design, reward_mode, training_seed)]
            for environment_seed in CONFIRMATION_EVALUATION_SEEDS:
                row = evaluate_episode(
                    agents, reward_mode, environment_seed, environment_config
                )
                row.update(
                    reward_design=design, reward_mode=reward_mode,
                    training_seed=training_seed, environment_seed=environment_seed,
                )
                confirmation_rows.append(row)
confirmation_episodes = pd.DataFrame(confirmation_rows)
print(f"Evaluated {len(confirmation_episodes)} confirmation episodes")

Evaluated 2400 confirmation episodes


### Compare outcomes and seed variation

- Each training seed is summarized over the same 20 fresh evaluation starts. The first table averages those seed summaries; the second shows their standard deviation.
- These are physical outcomes and action shares. I do not compare average rewards because the economic reward formulas have different scales.

In [80]:
confirmation_seed = confirmation_episodes.groupby(
    ["reward_design", "reward_mode", "training_seed"]
)[MEASURES].mean()
confirmation_mean = confirmation_seed.groupby(
    level=["reward_design", "reward_mode"]
).mean()
display(confirmation_mean[[
    "final_edi", "last_development_step", "mean_gpi",
    "peak_regional_pollution", "failure_percent",
    "mitigation_spending", "high_percent",
    "mean_incoming_spillover",
]].round(1))

final_edi  last_development_step  mean_gpi  \
reward_design reward_mode                                               
combined      global            75.2                   45.4      15.8   
              local             76.0                   46.2      13.1   
              mixed             76.5                   48.9      13.6   
edi_gain      global            65.2                   31.3       4.3   
              local             70.4                   28.7       2.7   
              mixed             70.1                   29.6       3.0   

                           peak_regional_pollution  failure_percent  \
reward_design reward_mode                                             
combined      global                          50.5              0.0   
              local                           44.6              0.0   
              mixed                           45.6              0.0   
edi_gain      global                          41.7              0.0   
              local                           41.1              0.0   
              mixed                           41.3              0.0   

                           mitigation_spending  high_percent  \
reward_design reward_mode                                      
combined      global                   22848.6          43.3   
              local                    24845.0          46.5   
              mixed                    24885.7          48.9   
edi_gain      global                   29270.7          58.5   
              local                    36016.7          87.1   
              mixed                    34040.2          78.1   

                           mean_incoming_spillover  
reward_design reward_mode                           
combined      global                           1.6  
              local                            1.2  
              mixed                            1.3  
edi_gain      global                           0.5  
              local                            0.3  
              mixed                            0.3

- Across the 20 new training seeds, the combined reward reaches final EDI 75-77, compared with 65-70 for EDI gain alone. The final development step also moves later, from about step 29-31 to step 45-49.
- Combined policies choose high mitigation less often and spend less on mitigation, but mean GPI rises from about 3-4 to 13-16. Incoming spillover also rises. More development therefore comes with a visible environmental cost.
- None of the 2,400 confirmation episodes fails. Mean peak regional pollution is about 45-51 for the combined reward, below the failure threshold of 100 on these starts.

In [81]:
confirmation_spread = confirmation_seed.groupby(
    level=["reward_design", "reward_mode"]
)[["final_edi", "mean_gpi", "mitigation_spending", "failure_percent"]].std()
display(confirmation_spread.round(1))

final_edi  mean_gpi  mitigation_spending  \
reward_design reward_mode                                             
combined      global             8.0       7.3               4670.7   
              local              5.2       4.6               3390.5   
              mixed              5.1       5.3               4309.1   
edi_gain      global            12.2       2.6               4497.4   
              local              2.9       0.5                610.3   
              mixed              5.8       0.9               2301.9   

                           failure_percent  
reward_design reward_mode                   
combined      global                   0.0  
              local                    0.0  
              mixed                    0.0  
edi_gain      global                   0.0  
              local                    0.0  
              mixed                    0.0

- These standard deviations are across 20 independent training seeds, after averaging each seed's evaluation starts. Combined global final EDI varies less than the EDI-gain reference (8.0 versus 12.2), while combined local varies more (5.2 versus 2.9).
- Combined GPI varies more across seeds in every sharing mode. The reward has not made policy behavior uniformly more stable.

### Compare matched training seeds

- The next table subtracts the EDI-gain result from the combined result within each reward mode and training seed. Positive EDI means the combined reward developed further; positive GPI means it allowed more pollution.
- The final table shows how often the combined design raised EDI or GPI across the 20 seeds. This is easier to read than a single mean when a few seeds behave differently.

In [82]:
confirmation_difference = (
    confirmation_seed.loc["combined"] - confirmation_seed.loc["edi_gain"]
)
difference_mean = confirmation_difference.groupby(level="reward_mode").mean()
difference_spread = confirmation_difference.groupby(level="reward_mode").std()
display(difference_mean[["final_edi", "mean_gpi", "failure_percent",
    "mitigation_spending", "high_percent", "mean_incoming_spillover"]].round(1))
display(difference_spread[["final_edi", "mean_gpi",
    "mitigation_spending"]].round(1))

,final_edi,mean_gpi,failure_percent,mitigation_spending,high_percent,mean_incoming_spillover
reward_mode,,,,,,
global,10.0,11.5,0.0,-6422.1,-15.2,1.1
local,5.6,10.4,0.0,-11171.7,-40.6,0.9
mixed,6.4,10.6,0.0,-9154.5,-29.2,1.0


,final_edi,mean_gpi,mitigation_spending
reward_mode,,,
global,12.9,8.0,6993.3
local,6.4,4.7,3336.7
mixed,9.2,5.5,5680.2


- On matched training seeds, the combined reward raises final EDI by 10.0 under global, 5.6 under local, and 6.4 under mixed. It raises mean GPI by 10.4-11.5 and incoming spillover by about 0.9-1.1 per region-step.
- Mitigation spending falls by about 6,400-11,200 per episode, and high mitigation choices fall by 15-41 percentage points. The higher GPI appears alongside much less mitigation, not just the extra emissions from higher EDI.
- The paired differences still vary across seeds. Final-EDI difference standard deviations are 12.9, 6.4, and 9.2 for global, local, and mixed; GPI difference standard deviations are 8.0, 4.7, and 5.5.

In [83]:
confirmation_direction = confirmation_difference.assign(
    more_edi=confirmation_difference["final_edi"] > 0,
    more_gpi=confirmation_difference["mean_gpi"] > 0,
)
direction_percent = (
    confirmation_direction.groupby(level="reward_mode")[["more_edi", "more_gpi"]]
    .mean() * 100
)
display(direction_percent.round(1))

,more_edi,more_gpi
reward_mode,,
global,75.0,90.0
local,65.0,100.0
mixed,60.0,100.0


- Combined reward raises final EDI in 15 of 20 global seeds, 13 of 20 local seeds, and 12 of 20 mixed seeds. The development improvement is common but not universal.
- GPI rises in 18 of 20 global seeds and all 20 local and mixed seeds. The pollution increase is the more consistent effect, so I should report it plainly rather than describe the combined reward as an unqualified improvement.

### Check each region

- I use the same fresh starts to check whether extra development and pollution are concentrated in one region. Each row below is a combined-minus-EDI-gain difference averaged within a seed, then across 20 seeds.
- A positive `mean_remaining_emissions` difference means that region sends more emissions to its neighbors. A positive `mean_incoming_spillover` difference means it receives more from them.

In [84]:
confirmation_region_rows = []
for design in CONFIRMATION_DESIGNS:
    for reward_mode in REWARD_MODES:
        for training_seed in CONFIRMATION_TRAINING_SEEDS:
            agents = confirmation_policies[(design, reward_mode, training_seed)]
            for environment_seed in CONFIRMATION_EVALUATION_SEEDS:
                rows = evaluate_regions(
                    agents, reward_mode, environment_seed, environment_config
                )
                for row in rows:
                    row.update(
                        reward_design=design, reward_mode=reward_mode,
                        training_seed=training_seed,
                    )
                confirmation_region_rows.extend(rows)
confirmation_regions = pd.DataFrame(confirmation_region_rows)
print(f"Recorded {len(confirmation_regions)} region-episodes")

Recorded 9600 region-episodes


In [85]:
CONFIRMATION_REGION_MEASURES = [
    "mean_pollution_load", "mean_remaining_emissions",
    "mean_incoming_spillover", "final_edi",
]
confirmation_region_seed = confirmation_regions.groupby(
    ["reward_design", "reward_mode", "training_seed", "region"]
)[CONFIRMATION_REGION_MEASURES].mean()
region_difference = (
    confirmation_region_seed.loc["combined"]
    - confirmation_region_seed.loc["edi_gain"]
)
display(region_difference.groupby(level=["reward_mode", "region"]).mean().round(2))

mean_pollution_load  mean_remaining_emissions  \
reward_mode region                                                  
global      0                     13.06                      3.29   
            1                     13.14                      3.42   
            2                      9.79                      2.14   
            3                      9.83                      2.11   
local       0                      7.86                      1.79   
            1                     11.62                      2.74   
            2                     12.54                      2.48   
            3                      9.66                      1.99   
mixed       0                     14.32                      3.56   
            1                     12.11                      2.71   
            2                      9.19                      1.96   
            3                      6.88                      1.73   

                    mean_incoming_spillover  final_edi  
reward_mode region                                      
global      0                          1.11      11.49  
            1                          1.08       3.64  
            2                          1.08       9.78  
            3                          1.11      14.93  
local       0                          1.04       3.00  
            1                          0.76      10.50  
            2                          0.76       2.92  
            3                          1.04       6.00  
mixed       0                          0.93      12.48  
            1                          1.06       9.00  
            2                          1.06       1.50  
            3                          0.93       2.55

- All four regions have higher average pollution loads and incoming spillover under the combined reward. The increases are uneven: under mixed reward, region 0's mean load rises by 14.32 while region 3's rises by 6.88.
- Each region also leaves more emissions after mitigation. Those emissions affect its neighbors in the same step, so the environmental cost is shared rather than confined to the region that develops or mitigates less.
- These are averages over seeds and starts. They show a regional burden worth discussing, not a reason to rank regions as inherently better or worse.

## Final reward decision

- I would keep **EDI gain plus ongoing income** as the working economic reward for the next Study 2 comparison, with the original emissions and mitigation settings. The 20-seed confirmation shows later and usually higher development in every sharing mode. I would keep local, global, and mixed as separate conditions and retain EDI gain alone as a reference.
- This choice does not mean the combined reward is environmentally better. It raises GPI and incoming spillover in most or all seeds, lowers mitigation spending, and places uneven costs on regions. None of the tested policies fails, so failure risk does not help separate the designs here.
- The useful finding is the economic-environmental tradeoff: ongoing income makes development more attractive, while the original reward encourages much more mitigation. I would report both outcomes and the seed and regional variation. I would not tune the income scale on these confirmation starts.